# Istanbul Healthcare Accessibility Analysis
## Part 1: Data Exploration

**Author:** Arife Mutlu  
**Date:** January 22, 2026  
**Goal:** Explore real healthcare facility distribution in Istanbul using OSM data

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('../src'))

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium

from load_data import load_osm_facilities, load_osm_districts
from buffer_analysis import create_buffers, coverage_percentage
from find_nearest import find_nearest

pd.set_option('display.max_columns', None)
%matplotlib inline

## 1. Load OSM Data
First run fetches from OpenStreetMap and caches locally; subsequent runs load from cache.

In [ ]:
facilities = load_osm_facilities()
districts = load_osm_districts()
print(f"Facilities: {len(facilities)}")
print(f"Districts:  {len(districts)}")
facilities.head()

## 2. Basic Exploration

In [ ]:
print(f"CRS: {facilities.crs}")
print(f"\nFacility types:")
print(facilities['facility_type'].value_counts())
print(f"\nSector breakdown:")
print(facilities['sector'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

facilities['facility_type'].value_counts().plot(kind='bar', ax=axes[0], color=['#e74c3c','#3498db','#2ecc71','#95a5a6'])
axes[0].set_title('Facility Type Distribution')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

facilities['sector'].value_counts().plot(kind='bar', ax=axes[1], color=['#f39c12','#8e44ad','#16a085','#7f8c8d'])
axes[1].set_title('Sector Distribution')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('../outputs/facility_distribution.png', dpi=150)
plt.show()

## 3. Interactive Facility Map

In [ ]:
color_map = {'Hospital': 'red', 'Clinic': 'blue', 'Doctor': 'green', 'Other': 'gray'}

m = folium.Map(location=[41.0082, 28.9784], zoom_start=10, tiles='CartoDB positron')

if districts is not None:
    folium.GeoJson(
        districts.__geo_interface__,
        style_function=lambda f: {'fillColor': '#f0f0f0', 'color': '#555', 'weight': 1, 'fillOpacity': 0.3}
    ).add_to(m)

for _, row in facilities.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6,
        popup=f"<b>{row['name']}</b><br>Type: {row['facility_type']}<br>Sector: {row['sector']}<br>District: {row['addr_district']}",
        color=color_map.get(row['facility_type'], 'gray'),
        fill=True, fillOpacity=0.7
    ).add_to(m)

m.save('../outputs/facility_map.html')
m

## 4. Buffer Analysis

In [ ]:
buffers = create_buffers(facilities, distances_km=[2, 5, 10])

for km, buf_gdf in buffers.items():
    pct = coverage_percentage(districts, buf_gdf)
    print(f"{km}km buffer → {pct:.1f}% of Istanbul area covered")

In [ ]:
m2 = folium.Map(location=[41.0082, 28.9784], zoom_start=10)

folium.GeoJson(
    buffers[5].__geo_interface__,
    style_function=lambda f: {'fillColor': 'blue', 'color': 'blue', 'fillOpacity': 0.2}
).add_to(m2)

for _, row in facilities.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4, color='red', fill=True, fillOpacity=0.8,
        popup=row['name']
    ).add_to(m2)

m2.save('../outputs/buffer_5km_map.html')
m2

## 5. Nearest Facility Analysis

In [ ]:
taksim = (28.9850, 41.0369)
nearest = find_nearest(taksim, facilities, k=5)
nearest['distance_km'] = (nearest['distance_m'] / 1000).round(2)
print("5 nearest facilities to Taksim Square:")
nearest[['name', 'facility_type', 'sector', 'distance_km']]

## Next Steps

- [ ] District-level accessibility scoring (facilities per km², distance-weighted)
- [ ] Road network travel-time analysis using osmnx + networkx
- [ ] Choropleth map of accessibility scores by district
- [ ] Identify underserved areas (low coverage + high population density)